In [ ]:
!pip install langchain langchain_openai langchain_community langchain-text-splitters langchain-core
pip install -U langchain langchain-community langchain-core langchain-text-splitters
pip install langchain-classic
pip install pypdf pandas openpyxl unstructured python-docx python-pptx

In [5]:
pip install langchain_chroma langchain_ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain_ollama]
Note: you may need to restart the kernel to use updated packages.


In [11]:
"""
Universal RAG Session Manager
==============================
Supports:
  - Incremental add/subtract of documents within a single session
  - Isolated parallel sessions (each gets its own Chroma collection)
  - Consistent naming convention: session_{session_id}

Naming convention for a given conversation:
  session_id  : "conv_<uuid4_short>" or caller-supplied
  collection  : "session_{session_id}"          ← Chroma collection name
  upload dir  : "uploads/{session_id}/"         ← where raw files land
"""

from __future__ import annotations

import os
import uuid
from pathlib import Path

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma                              # langchain-chroma >= 0.1
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader,
)
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings   # add this import at top
from langchain_ollama import ChatOllama
# ─────────────────────────────────────────────
# 1.  ENDPOINT CONFIGURATION
#     Point to your local provider (Ollama / vLLM / LM Studio)
# ─────────────────────────────────────────────

EMBEDDING_CONFIG = {
    "base_url":   "http://localhost:11434",
    "api_key":    "ollama",          # local providers usually ignore this
    "model_name": "embeddinggemma",
}

LLM_CONFIG = {
    "base_url":   "http://localhost:11434",
    "api_key":    "ollama",
    "model_name": "deepseek-r1:1.5b",
}

# Root directory that holds one Chroma collection folder per session
CHROMA_ROOT = "rag_sessions"

# Root directory for uploaded raw files (optional – only needed if you
# want to persist originals alongside the vector store)
UPLOADS_ROOT = "uploads"


# ─────────────────────────────────────────────
# 2.  HELPER – session-id factory
# ─────────────────────────────────────────────

def new_session_id() -> str:
    """Generate a short, human-readable session ID: conv_<8-hex-chars>."""
    return f"conv_{uuid.uuid4().hex[:8]}"


# ─────────────────────────────────────────────
# 3.  RAGSessionManager
# ─────────────────────────────────────────────

class RAGSessionManager:
    """
    One instance  = one isolated conversation session.

    Naming convention (everything derived from session_id):
        session_id          → e.g. "conv_a1b2c3d4"
        Chroma collection   → "session_conv_a1b2c3d4"
        Chroma persist dir  → "rag_sessions/session_conv_a1b2c3d4/"
        Upload directory    → "uploads/conv_a1b2c3d4/"
    """

    def __init__(self, session_id: str | None = None) -> None:
        # ── Identity ────────────────────────────────────────────────────
        self.session_id      = session_id or new_session_id()
        self.collection_name = f"session_{self.session_id}"
        self.chroma_dir      = os.path.join(CHROMA_ROOT, self.collection_name)
        self.upload_dir      = os.path.join(UPLOADS_ROOT, self.session_id)

        Path(self.chroma_dir).mkdir(parents=True, exist_ok=True)
        Path(self.upload_dir).mkdir(parents=True, exist_ok=True)

        print(f"[RAGSessionManager] session_id      = {self.session_id}")
        print(f"[RAGSessionManager] collection_name = {self.collection_name}")
        print(f"[RAGSessionManager] chroma_dir      = {self.chroma_dir}")

        # ── Models ──────────────────────────────────────────────────────
        # self._embeddings = OpenAIEmbeddings(
        #     model           = EMBEDDING_CONFIG["model_name"],
        #     openai_api_base = EMBEDDING_CONFIG["base_url"],
        #     openai_api_key  = EMBEDDING_CONFIG["api_key"],
        # )

        # in __init__, replace self._embeddings with:
        self._embeddings = OllamaEmbeddings(
            model          = EMBEDDING_CONFIG["model_name"],
            base_url       = EMBEDDING_CONFIG["base_url"],
        )

        self._llm = ChatOllama(
            model          = LLM_CONFIG["model_name"],
            base_url       = LLM_CONFIG["base_url"],
        )

        # ── Vector store (isolated collection per session) ───────────────
        # Each session gets its own collection name AND its own persist
        # directory, so two parallel sessions never touch each other's data.
        self._vectorstore = Chroma(
            collection_name   = self.collection_name,
            persist_directory = self.chroma_dir,
            embedding_function= self._embeddings,
        )

    # ────────────────────────────────────────
    # Internal: file loading
    # ────────────────────────────────────────

    def _load_file(self, file_path: str) -> list[Document]:
        ext = os.path.splitext(file_path)[-1].lower()
        loaders = {
            ".pdf":  PyPDFLoader,
            ".csv":  CSVLoader,
            ".txt":  TextLoader,
            ".yaml": TextLoader,
            ".yml":  TextLoader,
            ".xml":  TextLoader,
            ".doc":  UnstructuredWordDocumentLoader,
            ".docx": UnstructuredWordDocumentLoader,
            ".ppt":  UnstructuredPowerPointLoader,
            ".pptx": UnstructuredPowerPointLoader,
            ".xlsx": UnstructuredExcelLoader,
            ".xls":  UnstructuredExcelLoader,
        }
        loader_cls = loaders.get(ext)
        if loader_cls is None:
            print(f"[WARNING] Unsupported file type: {ext}")
            return []
        try:
            return loader_cls(file_path).load()
        except Exception as exc:
            print(f"[ERROR] Could not load {file_path}: {exc}")
            return []

    # ────────────────────────────────────────
    # Public: incremental ADD
    # ────────────────────────────────────────

    def add_document(self, file_path: str) -> int:
        """
        Chunk the file and embed it into THIS session's collection only.

        Returns the number of chunks added (0 on failure).
        """
        docs = self._load_file(file_path)
        if not docs:
            return 0

        filename = os.path.basename(file_path)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size    = 1_000,
            chunk_overlap = 100,
        )
        chunks = splitter.split_documents(docs)

        # Tag every chunk with session + source so we can delete by filename
        for chunk in chunks:
            chunk.metadata["session_id"] = self.session_id
            chunk.metadata["source"]     = filename

        self._vectorstore.add_documents(chunks)
        print(f"[{self.session_id}] Added {len(chunks)} chunks from '{filename}'")
        return len(chunks)

    # ────────────────────────────────────────
    # Public: incremental SUBTRACT
    # ────────────────────────────────────────

    def remove_document(self, filename: str) -> None:
        """
        Delete all chunks whose metadata matches (session_id, source).

        Chroma's .get() + .delete(ids=...) is the portable way to do this
        because the `where` parameter in .delete() is not available in all
        Chroma versions / backends.
        """
        # Step 1: fetch IDs for chunks from this file in this session
        results = self._vectorstore.get(
            where={
                "$and": [
                    {"session_id": {"$eq": self.session_id}},
                    {"source":     {"$eq": filename}},
                ]
            }
        )

        ids_to_delete = results.get("ids", [])
        if not ids_to_delete:
            print(f"[{self.session_id}] No chunks found for '{filename}' – nothing deleted.")
            return

        # Step 2: delete by explicit IDs (works in all Chroma versions)
        self._vectorstore.delete(ids=ids_to_delete)
        print(f"[{self.session_id}] Removed {len(ids_to_delete)} chunks for '{filename}'")

    # ────────────────────────────────────────
    # Public: list ingested files
    # ────────────────────────────────────────

    def list_documents(self) -> list[str]:
        """Return the unique filenames currently indexed in this session."""
        results   = self._vectorstore.get(where={"session_id": {"$eq": self.session_id}})
        sources   = {m.get("source") for m in results.get("metadatas", []) if m}
        filenames = sorted(s for s in sources if s)
        return filenames

    # ────────────────────────────────────────
    # Public: query
    # ────────────────────────────────────────

    def get_response(self, user_query: str) -> dict:
        """
        RAG inference scoped strictly to this session's collection.

        Returns the chain's full output dict; the answer is in ['answer'].
        """
        # Because each session has its own Chroma collection, the retriever
        # is already isolated – the metadata filter is a belt-and-suspenders
        # safety net for shared-collection deployments.
        retriever = self._vectorstore.as_retriever(
            search_kwargs={
                "k":      3,
                "filter": {"session_id": {"$eq": self.session_id}},
            }
        )

        system_prompt = (
            "You are a helpful assistant. Use the retrieved context below to "
            "answer the question. If the answer is not in the context, say you "
            "don't know.\n\nContext:\n{context}"
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human",  "{input}"),
        ])

        chain  = create_retrieval_chain(
            retriever,
            create_stuff_documents_chain(self._llm, prompt),
        )
        return chain.invoke({"input": user_query})

    # ────────────────────────────────────────
    # Dunder helpers
    # ────────────────────────────────────────

    def __repr__(self) -> str:
        return (
            f"RAGSessionManager("
            f"session_id={self.session_id!r}, "
            f"collection={self.collection_name!r})"
        )


In [9]:


# ─────────────────────────────────────────────
# 4.  USAGE EXAMPLES
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── Scenario A: incremental add / subtract within ONE session ────────

    print("\n=== Session A: incremental add/subtract ===")
    session_a = RAGSessionManager()               # auto-generates session_id

    session_a.add_document("/Users/himanshutiwari/Downloads/CV_Himanshu.pdf")
    #session_a.add_document("/path/to/financial_summary.csv")
    #session_a.add_document("/path/to/meeting_notes.txt")

    print("\nDocuments in session A:", session_a.list_documents())
    # e.g. ['financial_summary.csv', 'meeting_notes.txt', 'project_overview.pdf']

    # Realise meeting_notes.txt is irrelevant – remove it
    #session_a.remove_document("meeting_notes.txt")

    #print("\nAfter removal:", session_a.list_documents())
    # e.g. ['financial_summary.csv', 'project_overview.pdf']

    resp_a = session_a.get_response("Summarise the overview.")
    print("\nSession A answer:\n", resp_a["answer"])


    # # ── Scenario B: two PARALLEL sessions, fully isolated ───────────────

    # print("\n=== Session B & C: parallel isolation ===")

    # # Supply deterministic IDs so you can resume later
    # session_b = RAGSessionManager(session_id="conv_user_alice")
    # session_c = RAGSessionManager(session_id="conv_user_bob")

    # session_b.add_document("/path/to/alice_contract.pdf")
    # session_c.add_document("/path/to/bob_report.pdf")

    # # Session B can ONLY see alice_contract.pdf
    # resp_b = session_b.get_response("What are the contract terms?")
    # print("\nSession B answer:\n", resp_b["answer"])

    # # Session C can ONLY see bob_report.pdf
    # resp_c = session_c.get_response("What are the key findings?")
    # print("\nSession C answer:\n", resp_c["answer"])

    # # ── Scenario D: resume an existing session by ID ─────────────────────

    # print("\n=== Session D: resume session A ===")
    # session_a_resumed = RAGSessionManager(session_id=session_a.session_id)
    # print("Docs still in A:", session_a_resumed.list_documents())
    # # Chroma persisted to disk – data survives process restarts


=== Session A: incremental add/subtract ===
[RAGSessionManager] session_id      = conv_5b64a8f5
[RAGSessionManager] collection_name = session_conv_5b64a8f5
[RAGSessionManager] chroma_dir      = rag_sessions/session_conv_5b64a8f5
[conv_5b64a8f5] Added 6 chunks from 'CV_Himanshu.pdf'

Documents in session A: ['CV_Himanshu.pdf']


NotFoundError: 404 page not found

In [12]:
   
print("\n=== Session A: incremental add/subtract ===")
session_a = RAGSessionManager() 
session_a.add_document("/Users/himanshutiwari/Downloads/CV_Himanshu.pdf")
print("\nDocuments in session A:", session_a.list_documents())
resp_a = session_a.get_response("Summarise the overview.")
print("\nSession A answer:\n", resp_a["answer"])




=== Session A: incremental add/subtract ===
[RAGSessionManager] session_id      = conv_b549cde5
[RAGSessionManager] collection_name = session_conv_b549cde5
[RAGSessionManager] chroma_dir      = rag_sessions/session_conv_b549cde5
[conv_b549cde5] Added 6 chunks from 'CV_Himanshu.pdf'

Documents in session A: ['CV_Himanshu.pdf']

Session A answer:
 The context provides an overview of several key areas, including computational tools, programming languages, AI/ML advancements, data handling, visualization, specific collaborations, and a PhD researcher background. It also mentions associated technologies such as high-performance computing (HPC) systems and cloud platforms.


In [ ]:
"""
Universal RAG Session Manager  —  with RAG mode toggle
=======================================================

Modes:
  RAG_MODE  (opt-in)  : all queries grounded on uploaded documents
  CHAT_MODE (opt-out) : plain LLM chat, documents ignored

Transitions:
  CHAT_MODE  → RAG_MODE   : automatically on first add_document()
                             or manually via session.enable_rag()
  RAG_MODE   → CHAT_MODE  : session.disable_rag()
  CHAT_MODE  → RAG_MODE   : session.enable_rag()  (re-enable at any time)

Naming convention (all derived from session_id):
  session_id   →  conv_<8hex>
  collection   →  session_conv_<8hex>
  persist dir  →  rag_sessions/session_conv_<8hex>/
  upload dir   →  uploads/conv_<8hex>/
"""

from __future__ import annotations

import os
import uuid
from enum import Enum, auto
from pathlib import Path
from typing import Optional

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader,
)
from langchain_core.documents import Document


# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

EMBEDDING_MODEL = "nomic-embed-text:latest"
LLM_MODEL       = "deepseek-r1:1.5b"
OLLAMA_BASE_URL = "http://localhost:11434"

CHROMA_ROOT  = "rag_sessions"
UPLOADS_ROOT = "uploads"


# ─────────────────────────────────────────────
# SESSION MODE
# ─────────────────────────────────────────────

class SessionMode(Enum):
    CHAT = auto()   # plain LLM, no retrieval
    RAG  = auto()   # grounded on uploaded documents


# ─────────────────────────────────────────────
# PROMPTS
# ─────────────────────────────────────────────

CHAT_SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant.
Answer clearly and concisely using your general knowledge.
If you are unsure about something, say so honestly."""


def build_rag_system_prompt(loaded_files: list[str]) -> str:
    files_str = ", ".join(loaded_files) if loaded_files else "none"
    return f"""You are a precise document analysis assistant.
Your answers must be grounded exclusively in the retrieved context below.

## Loaded documents
{files_str}

## Rules
1. Only use the context below — never use outside knowledge.
2. Cite the source document name when answering.
3. If the context partially answers the question, give what you can and note what is missing.
4. If the answer is not in the context at all, say:
   "The provided documents do not contain information about [topic]."
5. Never infer, extrapolate, or fill gaps with plausible-sounding content.

## Context
{{context}}"""


# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────

def new_session_id() -> str:
    return f"conv_{uuid.uuid4().hex[:8]}"


# ─────────────────────────────────────────────
# RAGSessionManager
# ─────────────────────────────────────────────

class RAGSessionManager:
    """
    One instance = one isolated conversation session.

    Quick-start:
        session = RAGSessionManager()
        session.chat("Tell me about transformers")       # plain LLM chat
        session.add_document("report.pdf")               # auto-switches to RAG mode
        session.chat("Summarise the report")             # grounded on report.pdf
        session.disable_rag()
        session.chat("Tell me a joke")                   # back to plain LLM
        session.enable_rag()
        session.chat("What were the key findings?")      # grounded again
    """

    def __init__(self, session_id: Optional[str] = None) -> None:
        # ── Identity ────────────────────────────────────────────────────
        self.session_id      = session_id or new_session_id()
        self.collection_name = f"session_{self.session_id}"
        self.chroma_dir      = os.path.join(CHROMA_ROOT, self.collection_name)
        self.upload_dir      = os.path.join(UPLOADS_ROOT, self.session_id)

        Path(self.chroma_dir).mkdir(parents=True, exist_ok=True)
        Path(self.upload_dir).mkdir(parents=True, exist_ok=True)

        # ── Mode ────────────────────────────────────────────────────────
        self._mode: SessionMode = SessionMode.CHAT

        # ── Chat history (plain chat mode) ───────────────────────────────
        self._chat_history: list = []

        # ── Models ──────────────────────────────────────────────────────
        self._embeddings = OllamaEmbeddings(
            model    = EMBEDDING_MODEL,
            base_url = OLLAMA_BASE_URL,
        )
        self._llm = ChatOllama(
            model    = LLM_MODEL,
            base_url = OLLAMA_BASE_URL,
        )

        # ── Vector store (isolated per session) ─────────────────────────
        self._vectorstore = Chroma(
            collection_name    = self.collection_name,
            persist_directory  = self.chroma_dir,
            embedding_function = self._embeddings,
        )

        print(f"[Session {self.session_id}] Ready  |  mode = {self._mode.name}")

    # ────────────────────────────────────────
    # Mode controls
    # ────────────────────────────────────────

    @property
    def mode(self) -> SessionMode:
        return self._mode

    def enable_rag(self) -> str:
        """Switch to RAG mode (opt-in). Returns a user-facing status message."""
        docs = self.list_documents()
        if not docs:
            return (
                "No documents are loaded yet. "
                "Upload a document first — RAG mode will activate automatically."
            )
        self._mode = SessionMode.RAG
        msg = (
            f"RAG mode ON. All questions will now be answered based on: "
            f"{', '.join(docs)}.\n"
            f"Call disable_rag() to switch back to normal chat."
        )
        print(f"[Session {self.session_id}] {msg}")
        return msg

    def disable_rag(self) -> str:
        """Switch to plain chat mode (opt-out). Returns a user-facing status message."""
        self._mode = SessionMode.CHAT
        msg = (
            "RAG mode OFF. Switched to normal LLM chat — "
            "uploaded documents will be ignored until you re-enable RAG."
        )
        print(f"[Session {self.session_id}] {msg}")
        return msg

    def status(self) -> dict:
        """Return a snapshot of the current session state."""
        return {
            "session_id": self.session_id,
            "mode":       self._mode.name,
            "documents":  self.list_documents(),
        }

    # ────────────────────────────────────────
    # Document management
    # ────────────────────────────────────────

    def _load_file(self, file_path: str) -> list[Document]:
        ext = os.path.splitext(file_path)[-1].lower()
        loaders = {
            ".pdf":  PyPDFLoader,
            ".csv":  CSVLoader,
            ".txt":  TextLoader,
            ".yaml": TextLoader,
            ".yml":  TextLoader,
            ".xml":  TextLoader,
            ".doc":  UnstructuredWordDocumentLoader,
            ".docx": UnstructuredWordDocumentLoader,
            ".ppt":  UnstructuredPowerPointLoader,
            ".pptx": UnstructuredPowerPointLoader,
            ".xlsx": UnstructuredExcelLoader,
            ".xls":  UnstructuredExcelLoader,
        }
        loader_cls = loaders.get(ext)
        if loader_cls is None:
            print(f"[WARNING] Unsupported file type: {ext}")
            return []
        try:
            return loader_cls(file_path).load()
        except Exception as exc:
            print(f"[ERROR] Could not load {file_path}: {exc}")
            return []

    def add_document(self, file_path: str) -> str:
        """
        Ingest a file into this session's vector store.
        Automatically switches to RAG mode on the first document added.
        Returns a user-facing status message.
        """
        docs = self._load_file(file_path)
        if not docs:
            return f"Could not load '{file_path}'. Check the file path and format."

        filename = os.path.basename(file_path)
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks   = splitter.split_documents(docs)

        for chunk in chunks:
            chunk.metadata["session_id"] = self.session_id
            chunk.metadata["source"]     = filename

        self._vectorstore.add_documents(chunks)
        print(f"[{self.session_id}] Added {len(chunks)} chunks from '{filename}'")

        # ── Auto-switch to RAG mode on first upload ──────────────────
        first_upload = self._mode == SessionMode.CHAT
        self._mode   = SessionMode.RAG

        if first_upload:
            return (
                f"'{filename}' uploaded successfully ({len(chunks)} chunks indexed).\n\n"
                f"RAG mode is now ON — from this point all your questions will be "
                f"answered based on your uploaded documents.\n"
                f"To go back to normal chat at any time, call disable_rag()."
            )
        else:
            loaded = self.list_documents()
            return (
                f"'{filename}' added ({len(chunks)} chunks). "
                f"Active documents: {', '.join(loaded)}."
            )

    def remove_document(self, filename: str) -> str:
        """
        Remove all chunks for a specific file from this session.
        If no documents remain, automatically switches back to CHAT mode.
        Returns a user-facing status message.
        """
        results = self._vectorstore.get(
            where={"$and": [
                {"session_id": {"$eq": self.session_id}},
                {"source":     {"$eq": filename}},
            ]}
        )
        ids_to_delete = results.get("ids", [])

        if not ids_to_delete:
            return f"'{filename}' was not found in this session."

        self._vectorstore.delete(ids=ids_to_delete)
        print(f"[{self.session_id}] Removed {len(ids_to_delete)} chunks for '{filename}'")

        # ── Auto-switch back to CHAT if no docs remain ───────────────
        remaining = self.list_documents()
        if not remaining:
            self._mode = SessionMode.CHAT
            return (
                f"'{filename}' removed. No documents remain — "
                f"automatically switched back to normal chat mode."
            )

        return (
            f"'{filename}' removed. "
            f"Still active: {', '.join(remaining)}."
        )

    def list_documents(self) -> list[str]:
        """Return unique filenames currently indexed in this session."""
        results = self._vectorstore.get(where={"session_id": {"$eq": self.session_id}})
        sources = {m.get("source") for m in results.get("metadatas", []) if m}
        return sorted(s for s in sources if s)

    # ────────────────────────────────────────
    # Chat  (unified entry point)
    # ────────────────────────────────────────

    def chat(self, user_query: str) -> str:
        """
        Send a message. Behaviour depends on current mode:
          CHAT mode  →  plain LLM response with maintained history
          RAG mode   →  retrieval-grounded response
        """
        if self._mode == SessionMode.RAG:
            return self._rag_response(user_query)
        else:
            return self._chat_response(user_query)

    # ── Plain chat ───────────────────────────────────────────────────

    def _chat_response(self, user_query: str) -> str:
        """Standard multi-turn LLM chat with history."""
        self._chat_history.append(HumanMessage(content=user_query))

        messages = [SystemMessage(content=CHAT_SYSTEM_PROMPT)] + self._chat_history
        response = self._llm.invoke(messages)

        self._chat_history.append(AIMessage(content=response.content))
        return response.content

    # ── RAG response ─────────────────────────────────────────────────

    def _rag_response(self, user_query: str) -> str:
        """Retrieval-grounded response scoped to this session's documents."""
        retriever = self._vectorstore.as_retriever(
            search_kwargs={
                "k":      5,
                "filter": {"session_id": {"$eq": self.session_id}},
            }
        )

        # Guard: bail out before calling LLM if nothing was retrieved
        retrieved_docs = retriever.invoke(user_query)
        if not retrieved_docs:
            loaded = self.list_documents()
            if not loaded:
                return (
                    "No documents are loaded. "
                    "Upload a file or call disable_rag() to switch to normal chat."
                )
            return (
                f"The loaded documents ({', '.join(loaded)}) "
                f"do not contain relevant information about: '{user_query}'."
            )

        system_prompt = build_rag_system_prompt(self.list_documents())
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human",  "{input}"),
        ])

        chain = create_retrieval_chain(
            retriever,
            create_stuff_documents_chain(self._llm, prompt),
        )
        result = chain.invoke({"input": user_query})
        return result["answer"]

    # ── Backward-compat wrapper ──────────────────────────────────────

    def get_response(self, user_query: str) -> dict:
        """Backward-compatible wrapper around chat(). Prefer chat() for new code."""
        return {"answer": self.chat(user_query)}

    def __repr__(self) -> str:
        return (
            f"RAGSessionManager("
            f"session_id={self.session_id!r}, "
            f"mode={self._mode.name}, "
            f"docs={self.list_documents()})"
        )


# ─────────────────────────────────────────────
# USAGE EXAMPLES
# ─────────────────────────────────────────────

if __name__ == "__main__":

    session = RAGSessionManager()

    # ── 1. Normal chat before any upload ────────────────────────────
    print("\n--- Plain chat (no docs) ---")
    print(session.chat("What is a transformer model in ML?"))
    print(session.chat("Give me a simple Python example of a decorator."))
    print(session.status())
    # {'session_id': 'conv_xxxx', 'mode': 'CHAT', 'documents': []}

    # ── 2. Upload  →  auto-switches to RAG mode + user notice ───────
    print("\n--- Upload ---")
    print(session.add_document("/path/to/CV_Himanshu.pdf"))
    # "CV_Himanshu.pdf uploaded ... RAG mode is now ON ..."

    # ── 3. RAG-grounded questions ────────────────────────────────────
    print("\n--- RAG chat ---")
    print(session.chat("Summarise the document."))
    print(session.chat("What are the key skills mentioned?"))
    print(session.status())
    # {'mode': 'RAG', 'documents': ['CV_Himanshu.pdf']}

    # ── 4. Opt-out  →  plain LLM ────────────────────────────────────
    print("\n--- Opt-out ---")
    print(session.disable_rag())
    print(session.chat("Tell me a fun fact about space."))   # not grounded
    print(session.status())
    # {'mode': 'CHAT', ...}

    # ── 5. Opt back in ───────────────────────────────────────────────
    print("\n--- Opt back in ---")
    print(session.enable_rag())
    print(session.chat("What did the document say about experience?"))

    # ── 6. Add second doc, remove first ─────────────────────────────
    print("\n--- Incremental add/remove ---")
    print(session.add_document("/path/to/project_notes.txt"))
    print(session.remove_document("CV_Himanshu.pdf"))
    # auto-stays in RAG mode (project_notes.txt still active)

    print(session.remove_document("project_notes.txt"))
    # auto-switches back to CHAT mode (no docs remain)

    print(session.status())